In [ ]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import JsonOutputParser
from langchain_openai import ChatOpenAI
from pydantic import BaseModel, Field
from langchain_teddynote import logging
from dotenv import load_dotenv

load_dotenv()
logging.langsmith("CH03-OutputParser")

LangSmith 추적을 시작합니다.
[프로젝트명]
CH03-OutputParser


In [2]:
model = ChatOpenAI(temperature=0, model="gpt-5.6-luna")

In [3]:
class Topic(BaseModel): # 원하는 데이터 구조를 정의
    description: str = Field(description="주제에 대한 간결한 설명")
    hashtag: str = Field(description="해시태그 형식의 키워드(2개 이상)")

In [4]:
question = "치이카와의 귀여움에 대해 알려주세요. "

parser = JsonOutputParser(pydantic_object=Topic)
print(parser.get_format_instructions())

STRICT OUTPUT FORMAT:
- Return only the JSON value that conforms to the schema. Do not include any additional text, explanations, headings, or separators.
- Do not wrap the JSON in Markdown or code fences (no ``` or ```json).
- Do not prepend or append any text (e.g., do not write "Here is the JSON:").
- The response must be a single top-level JSON value exactly as required by the schema (object/array/etc.), with no trailing commas or comments.

The output should be formatted as a JSON instance that conforms to the JSON schema below.

As an example, for the schema {"properties": {"foo": {"title": "Foo", "description": "a list of strings", "type": "array", "items": {"type": "string"}}}, "required": ["foo"]} the object {"foo": ["bar", "baz"]} is a well-formatted instance of the schema. The object {"properties": {"foo": ["bar", "baz"]}} is not well-formatted.

Here is the output schema (shown in a code block for readability only — do not include any backticks or Markdown in your output):


In [5]:
prompt = ChatPromptTemplate.from_messages(
    [
        ("system", "당신은 귀여운 AI 어시스턴트 입니다. 질문에 귀엽게 답변하세요."),
        ("user", "#Format: {format_instructions}\n\n#Question: {question}"),
    ]
)

prompt = prompt.partial(format_instructions=parser.get_format_instructions())

chain = prompt | model | parser

answer = chain.invoke({"question": question})

In [6]:
answer["description"]

'치이카와는 작고 사랑스러운 모습과 순수하고 엉뚱한 매력으로 많은 사람의 마음을 사로잡는 캐릭터예요. 귀여운 표정과 친구들과의 따뜻한 우정이 특히 매력적이랍니다.'